# Function 5 · Week 10 decision review

Chemical-yield upper corner

[Detailed observations](observation_review.ipynb) · [Detailed fits](model_predictions.ipynb) · [Previous pooled study](full-cohort/model_predictions.ipynb) · [2024 comparison](../../../side-quests/historical-cohort-2024/README.md) · [Course data-use review](../../../docs/capstone-data-policy.md)

**Decision status.** Week 10 is prepared and unsubmitted. Round 9 is the latest official return. Start with the [prepared comparison](#prepared-week-10-decision) when choosing between the two portal batches.

## Expectations

Round 9 reduced x1 alone from the all-upper corner. Previous high scores suggested that moving off that corner would be costly.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

from IPython.display import display
import numpy as np
import pandas as pd

FUNCTION = 5
WEEK_DIR = Path.cwd().resolve()
WEEK = int(WEEK_DIR.name.removeprefix("Week_"))
STAGE = WEEK_DIR.parents[2]
SHARED = STAGE / "notebooks/Shared"
SPECIAL = STAGE / "notebooks/Function_1/FunctionSpecificShared"
sys.path[:0] = [str(SHARED), str(SPECIAL)]
from weekly_peak_review import peak_review, portal_point
from query_budget import remaining_queries

analysis = WEEK_DIR / "official" if FUNCTION <= 4 else WEEK_DIR
official_path = analysis / "observations.csv"
model_snapshot = json.loads((analysis / "model-review.json").read_bytes())
observation_snapshot = json.loads((analysis / "observation-review.json").read_bytes())
feedback = json.loads((STAGE / f"submissions/round-{WEEK-1:02d}-results.json").read_bytes())
record = feedback["results"][FUNCTION - 1]
forecast = observation_snapshot["historical_forecast"]
if (model_snapshot["week"], model_snapshot["function"], model_snapshot["snapshot_sha256"]) != (
        WEEK, FUNCTION, sha256(official_path.read_bytes()).hexdigest()):
    raise ValueError("The official model review and observations describe different snapshots")
if (feedback["round"], record["function"], record["portal_input"], record["output"]) != (
        WEEK - 1, FUNCTION, forecast["portal_input"], forecast["actual_output"]):
    raise ValueError("The saved forecast must match the returned observation")
official = pd.read_csv(official_path, float_precision="round_trip")
cols = [c for c in official if c.startswith("x")]
x = official[cols].to_numpy(float)
raw_y = official.y.to_numpy(float)
later = official.source_round.to_numpy() > 0
best_row = official.loc[official.y.idxmax()]
incumbent = best_row[cols].to_numpy(float)
queries_left = remaining_queries(WEEK_DIR)
units = forecast["prediction_units"]
target = raw_y
if FUNCTION == 1:
    from function_1_model_helpers import signed_log
    target = signed_log(raw_y, scale=forecast["height_scale"])

display(pd.Series({
    "Archived forecast source": forecast["source"],
    "Submitted input": forecast["portal_input"],
    "Forecast units": units,
    "Saved GP mean": forecast["gp_mean"],
    "GP latent SD": forecast["gp_latent_sd"],
    "Saved neural mean": forecast["neural_mean"],
    "Neural seed SD": forecast["neural_seed_sd"],
}, name="Forecast frozen before the return"))

Archived forecast source    notebooks/Function_5/Week_09/figures/neural-ne...
Submitted input                           0.976212-0.999999-0.999999-0.999999
Forecast units                                                       output y
Saved GP mean                                                      8309.66074
GP latent SD                                                       140.191201
Saved neural mean                                                 8303.712978
Neural seed SD                                                       1.699938
Name: Forecast frozen before the return, dtype: object

## Results

The x1 reduction cost about 439 points. The confirmed corner score matches the strongest legal portal result checked in the 2024 archive. Controlled one-coordinate reductions from the corner help distinguish boundary behaviour from a coincidental high return.

In [2]:
latest_fitted = forecast["actual_in_prediction_units"]
display(pd.Series({"Returned score": record["output"], "Best before return": record["best_before"],
                   "Best now": record["best_after"], "Return rank": observation_snapshot["latest_rank"],
                   "Actual in forecast units": latest_fitted,
                   "Actual minus GP mean": forecast["actual_minus_gp_mean"],
                   "Actual minus neural mean": forecast["actual_minus_neural_mean"]}, name="Official result"))

# These references describe observed outcomes. The decision fits use official rows.
def optional_reference(folder, file, audit_name, expected):
    path = folder / file
    manifest = folder / audit_name
    if not (path.is_file() and manifest.is_file()):
        return None
    manifest_data = json.loads(manifest.read_bytes())
    if sha256(path.read_bytes()).hexdigest() != expected(manifest_data):
        raise ValueError(f"Reference hash mismatch: {path}")
    return pd.read_csv(path, float_precision="round_trip")

current_reference = optional_reference(
    STAGE / "data/cohort/2026-11-25", f"function_{FUNCTION}/observations.csv", "dataset-audit.json",
    lambda a: a["summary"][FUNCTION - 1]["sha256"],
)
archive = optional_reference(
    STAGE / "data/historical-cohort/2024", f"function_{FUNCTION}.csv", "intake-audit.json",
    lambda a: a["functions"][str(FUNCTION)]["sha256"],
)
reference_rows = []
if current_reference is not None:
    peer = current_reference.loc[current_reference.observation_id.str.startswith("peer-")]
    reference_rows.append(("Published current cohort", peer))
if archive is not None:
    reference_rows.append(("2024 archive, including extra starters", archive))
benchmarks = []
for label, frame in reference_rows:
    row = frame.loc[frame.y.idxmax()]
    benchmarks.append({"Scope": label, "Highest measured": row.y,
                       "Difference from your best": row.y - best_row.y,
                       "Distance to your best": np.linalg.norm(row[cols].to_numpy(float) - incumbent)})
if benchmarks:
    display(pd.DataFrame(benchmarks).round(8))
else:
    print("Local comparison snapshots are unavailable for this run.")

Returned score              8223.209090
Best before return          8662.405001
Best now                    8662.405001
Return rank                    6.000000
Actual in forecast units    8223.209090
Actual minus GP mean         -86.451650
Actual minus neural mean     -80.503888
Name: Official result, dtype: float64

,Scope,Highest measured,Difference from your best,Distance to your best
0,Published current cohort,6947.430895,-1714.974106,0.110027
1,"2024 archive, including extra starters",8662.405001,0.000000,0.000000


In [3]:
changed = np.count_nonzero(x.round(6) != incumbent.round(6), axis=1)
face = official.loc[changed == 1, ["observation_id", *cols, "y"]].copy()
face["Change from corner"] = face.y - best_row.y
display(face.sort_values("y", ascending=False))
print(f"Observed upper corner: {portal_point(incumbent)}, output {best_row.y:.9f}.")

,observation_id,x1,x2,x3,x4,y,Change from corner
24,round-05,0.999999,0.999999,0.999999,0.979636,8283.828104,-378.576898
26,round-07,0.999999,0.999999,0.979593,0.999999,8283.061544,-379.343457
28,round-09,0.976212,0.999999,0.999999,0.999999,8223.209090,-439.195911
27,round-08,0.999999,0.969053,0.999999,0.999999,8099.179959,-563.225042


Observed upper corner: 0.999999-0.999999-0.999999-0.999999, output 8662.405001248.


## Predictions

**Local peak and global evidence**

The corner is the strongest candidate for a near-global result among the functions without an exact structural fit. High finite-set support is conditional on the smooth models and 100-point tolerance. Global monotonicity has not been established by the brief or the measurements. The predictive error across the nine official returns is much larger than the working tolerance, so the displayed draw fractions support a careful check of the corner.

### Prepared Week 10 decision

**Prepared and unsubmitted as of 25 November 2026.** Round 9 is the latest confirmed feedback. Four portal queries remain per function, including Week 10. Use the cohort-informed route only if the course permits that use, including the exact published F2 repeat. The official-data fallback is ready if the rules remain unclear near the portal deadline.

Both batches keep the single-axis x4 experiment at the upper corner. It clears the 0.020 spacing rule and the official surrogate expects a lower score than the 8,662.405 incumbent. The useful outcome is a measurement of the corner slope and its persistence at the larger step.

**Week 11 depends on the return.** Use the result to decide whether the corner loss is consistent with previous probes. Spend further queries where they provide evidence beyond another step down the same arm.

[Week 10 course check and exact batches](../../../submissions/Week_10/README.md) · [Cohort advice and replay](../../../submissions/Week_10/advisory-review.md)

The estimates below use the advisory review's official-data GP comparator and predictive return standard deviation, which includes modelled observation noise. F8 uses the validated starter quadratic and omits a standard deviation because that fit supplies no probabilistic interval. The rows are proposed inputs with model estimates.

In [4]:
from week_10_decision_record import decision_comparison

with pd.option_context("display.max_colwidth", None):
    display(decision_comparison(STAGE, FUNCTION, official))

,Portal input,Official GP mean (score),Predictive return SD
Stage,,,
Initial proposal,0.999999-0.999999-0.999999-0.959224,8249.396467,162.664087
Advisory experiment,0.999999-0.999999-0.999999-0.959224,8249.396467,162.664087
Revised cohort-informed,0.999999-0.999999-0.999999-0.959224,8249.396467,162.664087
Official-data fallback,0.999999-0.999999-0.999999-0.959224,8249.396467,162.664087


### Conditional peak check

Two Gaussian processes with fitted noise share the same candidate set. A local candidate lies within the stated Euclidean radius of the observed incumbent. The columns report the fraction of joint latent draws in which none of the candidates in that set beats the incumbent by more than the working tolerance. The broader search includes Sobol points, corners, observed coordinates and surrogate-suggested probes. It cannot certify the continuous function or rule out unrepresented peaks. The nine grouped official return holdouts and nominal 90% observation-band counts indicate predictive fit. They do not calibrate the global-maximum event. The tolerance and radius are analysis choices.

In [5]:
radius = 0.075
tolerance = 100
peak = peak_review(x, target, later, radius=radius, tolerance=tolerance,
                   length_scales=[0.32, 0.32, 0.32, 0.32], noise=[0.001, 1e-08, 0.1],
                   minimum_distance=model_snapshot["minimum_query_distance"])
print(f"Local radius {radius:g}. Score tolerance {tolerance:g} in {units}.")
print(f"Joint posterior over {peak['candidate_count']} candidates, including {peak['local_count']} local."
      f" {peak['probability_draws']} draws per kernel.")
display(peak["summary"].round(5))
print("Adaptive two-query comparison on the Matérn shortlist")
one_query = peak["acquisitions"]["Matérn 3/2"].nlargest(1, "One-query EI")
print(f"Best one-query input: {one_query.iloc[0]['Input']}")
display(peak["acquisitions"]["Matérn 3/2"].head(5).round(6))

Local radius 0.075. Score tolerance 100 in output y.
Joint posterior over 427 candidates, including 135 local. 1024 draws per kernel.


,Local set within tolerance,Broader set within tolerance,Broader gap median,Broader gap 90th pct,Return MAE,Return RMSE,In 90% observation band,Return holdouts,Fitted noise SD,Fit warnings
Model,,,,,,,,,,
Matérn 3/2,0.99414,0.99414,0.0,0.0,404.64533,634.07216,7,9,122.09698,3
RBF,1.00000,1.00000,0.0,0.0,518.81286,955.62996,6,9,111.63505,0


Adaptive two-query comparison on the Matérn shortlist
Best one-query input: 0.999999-0.999999-0.999853-0.958574


,Input,Region,Mean,Observed SD,One-query EI,Two-query gain
0,0.999999-0.999999-0.999853-0.958574,Local,8245.038091,163.577895,0.279175,0.565356
1,0.999999-0.999999-0.999999-0.956158,Local,8232.363886,167.154603,0.266819,0.555509
3,0.999999-0.999999-0.999999-0.926499,Local,8063.140073,218.246893,0.198992,0.545671
4,0.999999-0.999999-0.934895-0.999999,Local,8099.592249,202.743442,0.167219,0.446238
5,0.999999-0.999999-0.926499-0.999999,Local,8049.395994,218.211310,0.160996,0.440147


### Strategy for the remaining queries

Compare the sign of local face slopes and the monotonicity assumption before relying on a generic GP. If monotonicity throughout the full box were known, the upper corner would be the constrained global maximiser. A targeted perturbation tests that assumption locally. Monotone Gaussian processes are a shape-constrained option if directional evidence warrants the constraints.

The code compares one-query expected improvement in the returned score with a two-query approximation. It conditions on a first outcome, then chooses a different point on a shortlist. Both steps apply the saved minimum-spacing rule. It holds fitted kernel parameters fixed. Refit after each actual result and reassess the next action. The four rounds below are conditional decisions.

| Round | Action given the evidence so far |
| --- | --- |
| 10 | Test one larger x4 step away from the upper corner using the checked spacing. Both prepared routes use the same input. |
| 11 | Use the result to decide whether the corner loss is consistent with previous probes. Spend further queries where they provide evidence beyond another step down the same arm. |
| 12 | Choose the highest-value remaining boundary test, including an off-corner control only if an alternative model predicts a material gain. |
| 13 | Use the best empirically supported near-corner input that can still improve the recorded score. |

[Jones, Schonlau and Welch, efficient global optimisation](https://www.schonlau.net/publication/jogo98.pdf) · [Frazier, Bayesian optimisation tutorial](https://arxiv.org/abs/1807.02811) · [Riihimäki and Vehtari, monotone Gaussian processes](https://proceedings.mlr.press/v9/riihimaki10a.html)

In [6]:
print(f"{queries_left} query opportunities per function, including Week {WEEK}.")
destination = WEEK_DIR / "weekly-peak-review.json"
summary = {"function": FUNCTION, "week": WEEK, "queries_remaining": queries_left,
           "official_snapshot_sha256": model_snapshot["snapshot_sha256"],
           "candidate_count": peak["candidate_count"], "local_count": peak["local_count"],
           "probability_draws": peak["probability_draws"], "local_radius": radius,
           "tolerance": tolerance, "target_units": units,
           "models": peak["summary"].reset_index().to_dict(orient="records"),
           "matérn_shortlist": peak["acquisitions"]["Matérn 3/2"].to_dict(orient="records")}
if FUNCTION == 8:
    summary["structural_fit"] = quadratic
destination.write_bytes((json.dumps(summary, indent=2, allow_nan=False) + "\n").encode())

4 query opportunities per function, including Week 10.


3710

### Course lifecycle

1. **Define the goal and scope** – choose the next official query while assessing how close the incumbent might be to the peak.
2. **Gather the data** – preserve the official snapshot, the frozen predictions and labelled published comparisons.
3. **Explore the data** – compare the returned result with local and external observations.
4. **Clean and preprocess the data** – verify hashes and portal alignment. Preserve the recorded raw score.
5. **Select and engineer features** – use the original coordinates, with a signed-log target for F1. Structural fits declare their terms separately.
6. **Define the machine-learning task** – regression supports maximisation and assessment of the incumbent.
7. **Partition the data** – withhold each official return and any matching portal coordinate for the predictive check.
8. **Select and train candidate methods** – compare two kernel assumptions and function-specific hypotheses.
9. **Evaluate and interpret the results** – judge forecast reliability and the finite-domain comparison before choosing how to spend the remaining budget.
10. **Deploy and iterate** – deployment is not applicable. The next portal return will update the review and the query plan.